In [1]:
import gdstk


In [24]:
#JJ principal del fluxonium
# bridge_width = 0.54
# DIMS = dict(
#     wa = 1.04, ha = 1.57, 
#     wb=0.9, hb= 0.35, 
#     wc= 2, hc = 1.05,
#     wd = 1.40, hd = 0.35,
#     we = bridge_width, he=2, 
#     wf = 0.44, hf = 0.35, 
#     wg = bridge_width+0.05+0.05, hg = 0.15 ,
#     wh= 1.41, hh= 0.35 ,
#     wi=1.70 , hi = 0.25 ,
#     wj=0.8 , hj= 1.68, 
#     wk = 1.70, hk = 0.35,
#     gap = 0.05,
# )


In [2]:
# JJ del transmon


bridge_width = 0.938
DIMS = dict(
    wa = 1.04, ha = 1.57, 
    wb=1.27, hb= 0.35, 
    wc= 2.3, hc = 1.05,
    wd = 1.28, hd = 0.35,
    we = bridge_width, he=2, 
    wf = 0.44, hf = 0.35, 
    wg = bridge_width+0.05+0.05, hg = 0.15 ,
    wh= 1.3 , hh= 0.35 ,
    wi=2.15 , hi = 0.25 ,
    wj=0.61 , hj= 1.68, 
    wk = 2.15 , hk = 0.35,
    gap = 0.05,
)


In [3]:
# PARAMETROS PARA LAS JUNTURAS DEL ARRAY


bridge_width = 2.68
DIMS = dict(
    wa = 1.04, ha = 1.57, 
    wb=3.10, hb= 0.35, 
    wc= 4.14, hc = 1.05,
    wd = 1.5, hd = 0.35,
    we = bridge_width, he=2, 
    wf = 0.44, hf = 0.35, 
    wg = bridge_width+0.05+0.05, hg = 0.15 ,
    wh= 1.5 , hh= 0.35 ,
    wi=3.59 , hi = 0.25 ,
    wj=1.04 , hj= 2.5, 
    wk = 3.58 , hk = 0.35,
    gap = 0.05,
)

In [4]:


LAYER_BASE = 1   # azul
LAYER_TOP = 2    # rojo


def build_unit_cell(dims=DIMS, x0=0.0, y0=0.0):
    """
    Construye UNA unidad (una juntura) como lista de gdstk.Polygon,
    separadas en capa base y capa top. x0,y0 es la posicion del
    techo de la celda (donde arranca el pad rojo A).

    Devuelve: (polys_base, polys_top, cell_height)
    """
    d = dims
    polys_base = []
    polys_top = []

    def add(bucket, x_left, y_cursor, w, h):
        """
        x_left   : borde izquierdo del rectangulo (relativo a x0)
        y_cursor : distancia desde el TECHO de la celda hasta el
                   borde SUPERIOR del rectangulo (positivo = mas abajo)
        """
        x_abs0 = x0 + x_left
        x_abs1 = x_abs0 + w
        y_top = y0 - y_cursor
        y_bot = y_top - h
        bucket.append(gdstk.rectangle((x_abs0, y_bot), (x_abs1, y_top)))


    cursor = 0.0

    # A
    add(polys_top, 0.0, cursor, d['wa'], d['ha'])

    # B
    cursor+= d['ha']
    xb =   - d['gap'] - d['wb']
    add(polys_base, xb , cursor- d['gap'] - d['hb'], d['wb'], d['hb'])

    #C
    xc = - abs(d['wc']-d['wa'])
    add(polys_top, xc, cursor, d['wc'], d['hc'])

    #E                      ####ESTE ES LA PARTE SUPERIOR DEL PUENTE
    cursor += d['hc']
    xe= xc
    add(polys_top, xe, cursor, d['we'], d['he'])

    #D
    xd = xe + d['we'] + d['gap']
    add(polys_base, xd, cursor + d['gap'], d['wd'], d['hd'])

    #F
    cursor += d['he']
    xf = xe - d['gap'] - d['wf']
    add(polys_base, xf , cursor + d['gap'] + d['hg'] - d['hf'] , d['wf'], d['hf'])

    #G
    cursor += d['gap']
    xg = xf + d['wf']
    add(polys_base, xg, cursor, d['wg'], d['hg'])

    #H 
    cursor+= d['hg']
    xh= xg + d['wg']
    add(polys_base, xh, cursor - d['hh'], d['wh'], d['hh'])
 

    #I
    cursor += d['gap']
    xi = xf 
    add(polys_top, xi , cursor, d['wi'], d['hi'])

   #K
    cursor += d['hi'] + d['gap']
    xk = xi - d['gap']
    add(polys_base, xk, cursor, d['wk'], d['hk'])

    #J
    cursor = cursor - d['gap'] - d['hi']
    xj = xi + d['wi']
    add(polys_top, xj , cursor, d['wj'], d['hj'])

 

    cell_height = cursor + d['hj']

    return polys_base, polys_top, cell_height


# =====================================================================
# PASO 2: encadenar N unidades (el array completo)
# =====================================================================
def build_chain(n_units, dims=DIMS, x0=0.0, y0=0.0, pitch=None):
    """
    Devuelve (polys_base, polys_top) con las N unidades apiladas
    verticalmente. pitch=None -> se calcula automaticamente.
    """
    _, _, auto_pitch = build_unit_cell(dims, 0, 0)
    if pitch is None:
        pitch = auto_pitch

    all_base, all_top = [], []
    for i in range(n_units):
        y_i = y0 - i * pitch
        pb, pt, _ = build_unit_cell(dims, x0, y_i)
        all_base.extend(pb)
        all_top.extend(pt)
    return all_base, all_top, pitch


# =====================================================================
# PASO 3: crear la libreria GDS y exportar
# =====================================================================
def export_gds(filename, n_units=1, dims=DIMS):
    lib = gdstk.Library()
    cell = lib.new_cell("JUNCTION_ARRAY")

    polys_base, polys_top, pitch = build_chain(n_units, dims)

    for p in polys_base:
        p.layer = LAYER_BASE
        cell.add(p)
    for p in polys_top:
        p.layer = LAYER_TOP
        cell.add(p)

    lib.write_gds(filename)
    print(f"GDS exportado: {filename}  ({n_units} unidad(es), pitch={pitch:.4f} um)")
    return filename


if __name__ == "__main__":
    # --- UNA sola juntura ---
    # export_gds(r"C:\Users\Usuario\Documents\TESIS\PROGRAMAS\GDS\single_jj_transmon.gds", n_units=1)

    # --- Cadena de 80junturas ---
    export_gds(r"junction_chain_100_larger.gds", n_units=100)

GDS exportado: junction_chain_100_larger.gds  (100 unidad(es), pitch=7.3700 um)
